# Lilly — the voice pipeline's control (`speak-control`)

**Job:** `speak-control`. Pre-registered in `training/PREREGISTRATION.md`, "v7 — speak
— the control, written before any run". **Read it before launching.**

**The question.** Two voices fine-tuned from Piper's `sr_RS-serbski_institut-medium`
— on FLEURS (v5) and on the Croatian parliament (v6) — were heard near 52% where
the checkpoint itself is heard at 22.3%. Does the recipe preserve a voice it is
handed? This run fine-tunes the checkpoint on **its own 747 training utterances**
(Sorbian Institute, CC BY-NC-SA 4.0, public; the checkpoint repository carries the
list) for 3,000 steps in three arms — `sr` phonemes, `bs` phonemes, `bs` with a
gentle optimizer — exports speaker 0 of each, and judges them on the FLEURS test
prefix beside the before voice and the human recordings.

**Attach, before Save & Run All** (`scripts/kaggle_train.py speak-control` does it):
**Add data → Datasets → `lilly-listen-large-v3`** (the shipped listener, `e6bb58483586b06c`,
**required**). Internet **On**: GitHub releases (the two FLACs), FLEURS test, pip, Hugging Face.

**Output.** `lilly-speak-control-results.zip` after the judgment, whichever way it
fell. **No voice zip, ever**: nothing trained here is Bosnian and nothing ships.
ERROR or CANCEL: nothing here is a result.


In [ ]:
# 1. Stop here unless the machine is actually set up
import json, os, shutil, subprocess, sys, urllib.error, urllib.request, zipfile
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
GPU = torch.cuda.get_device_name(0)
print(torch.cuda.device_count(), "GPU(s) visible, using:", GPU)

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass
    except Exception as exc:
        raise SystemExit(f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")
for host in ("https://github.com", "https://pypi.org", "https://huggingface.co",
             "https://datasets-server.huggingface.co"):
    reachable(host)
print("network ok")

# The child's stdout is NOT the Kaggle log. Tee everything into Output so a run
# whose trainer printed nothing cannot be mistaken for one that trained.
TEE = Path("/kaggle/working/stdout.txt")
TEE.parent.mkdir(parents=True, exist_ok=True)

def run(*cmd, quiet=False, env=None):
    line = "$ " + " ".join(str(c) for c in cmd)
    print(line, flush=True)
    with TEE.open("a", encoding="utf-8") as sink:
        sink.write(line + "\n")
        child = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1,
                                 env={**os.environ, **(env or {})})
        for out in child.stdout:
            if not quiet:
                print(out, end="", flush=True)
            sink.write(out)
        code = child.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)


In [ ]:
# 2. Get the Lilly code -- into scratch, never into Output
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
subprocess.run(["rm", "-rf", str(CLONE)], check=True)
os.chdir(SCRATCH)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert (CLONE / "training").is_dir(), "clone produced nothing"
os.chdir(CLONE)
print("working in", os.getcwd())

sys.path.insert(0, str(CLONE))
from training.kaggle_offload import Offload
OFF = Offload("speak-control", os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "manual"))
OFF.hardware(GPU)
print("offload log started (experiment_log.json, metrics.jsonl):", OFF.body["git"])


In [ ]:
# 3. Install (~3 min): Piper with its training extras, the speaker encoder, the listener.
# Not torch: Kaggle's CUDA build stays, and a pip that replaced it would put the
# run on a card the trainer cannot see. Checked in a fresh interpreter afterwards,
# because the torch already imported in this one would hide the swap.
# onnxscript: torch's ONNX exporter imports it, and piper's train extra does not list it.
NEEDED = ["piper-tts[train]", "onnxscript", "resemblyzer", "setuptools<81", "scikit-learn",
          "faster-whisper", "soundfile", "huggingface_hub", "pyarrow"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
wanted = [pins.get(n.split("[")[0].split("<")[0].lower(), n) for n in NEEDED]
print("installing:", wanted)
TORCH_BEFORE = torch.__version__
run(sys.executable, "-m", "pip", "install", "-q", *wanted)
probe = subprocess.run([sys.executable, "-c",
                        "import torch, lightning, piper, resemblyzer, faster_whisper; "
                        "print(torch.__version__, torch.cuda.is_available(), lightning.__version__)"],
                       text=True, capture_output=True)
print(probe.stdout.strip() or probe.stderr[-1200:])
fields = probe.stdout.split()
if probe.returncode or len(fields) < 3 or fields[0] != TORCH_BEFORE or fields[1] != "True":
    raise SystemExit(f"after pip: {fields[:2]} against torch {TORCH_BEFORE} before -- pip replaced "
                     "torch or an import fails; not training on a card the trainer cannot see")
# resemblyzer drags in the 2015 `typing` backport, which shadows the standard
# library's whenever the working directory is site-packages. Gone, before it can.
run(sys.executable, "-m", "pip", "uninstall", "-y", "-q", "typing")


In [ ]:
# 3b. Piper's monotonic alignment is a Cython extension the wheel does not
# carry built: piper1-gpl ships core.pyx in its source tree and a setup.py that
# expects it beside __init__.py, and the trainer imports the built module from a
# nested package. Fetched at the 1.8.0 tag, pinned by sha256, built into scratch,
# copied where the import looks. Without it training stops at the first batch,
# so a failed import stops the run here instead.
import hashlib
import piper
PYX_URL = ("https://raw.githubusercontent.com/OHF-Voice/piper1-gpl/v1.8.0/"
           "src/piper/train/vits/monotonic_align/core.pyx")
PYX_SHA = "8640b303683823a4a1259179547ef476999b1cbb2e46ff656b970763cfbc1157"
MA = Path(piper.__file__).parent / "train" / "vits" / "monotonic_align"
inner = MA / "monotonic_align"
inner.mkdir(exist_ok=True)
(inner / "__init__.py").touch()
pyx = urllib.request.urlopen(PYX_URL, timeout=60).read()
if hashlib.sha256(pyx).hexdigest() != PYX_SHA:
    raise SystemExit("core.pyx from GitHub is not the 1.8.0 file the pre-registration pinned")
(inner / "core.pyx").write_bytes(pyx)
BUILD = SCRATCH / "ma-build"
run(sys.executable, "-c",
    "import sys, numpy; from setuptools import setup; from Cython.Build import cythonize; "
    "setup(name='monotonic_align', script_args=['build_ext', '--build-lib', sys.argv[1], "
    "'--build-temp', sys.argv[2]], ext_modules=cythonize(sys.argv[3], language_level=3), "
    "include_dirs=[numpy.get_include()])",
    str(BUILD / "lib"), str(BUILD / "tmp"), str(inner / "core.pyx"), quiet=True)
built_so = sorted((BUILD / "lib").rglob("core*.so"))
if len(built_so) != 1:
    raise SystemExit(f"expected one built core extension, found {built_so}")
shutil.copy(built_so[0], inner / built_so[0].name)
probe = subprocess.run([sys.executable, "-c",
                        "from piper.train.vits.monotonic_align import maximum_path; print('ok')"],
                       text=True, capture_output=True)
if "ok" not in probe.stdout:
    raise SystemExit("monotonic_align still does not import:\n" + probe.stderr[-1500:])
print("monotonic_align built:", built_so[0].name)


In [ ]:
# 4. Smoke, before anything heavy: the card computes, and espeak-ng's bs voice is
# reachable from Piper's bundled data. (The wheel's data path breaks past ~160
# characters -- seen on the Mac -- so it is checked where the run will read it.)
x = torch.randn(256, 256, device="cuda", requires_grad=True)
(x @ x).sum().backward()
assert x.grad is not None and torch.isfinite(x.grad).all(), "CUDA backward failed"
from piper.phonemize_espeak import EspeakPhonemizer
phonemes = EspeakPhonemizer().phonemize("bs", "Sastanak je 2026. godine, dvije žene.")
flat = "".join("".join(s) for s in phonemes)
print("espeak bs:", flat)
if "dvˈije" not in flat or "ʒ" not in flat:
    raise SystemExit(f"espeak-ng bs did not phonemize as expected: {flat!r}")
print("smoke ok")


In [ ]:
# 5. Three things, each checked. (a) FLEURS bs_ba test, whole: its first 200
# clips judge. (b) The voice's own training list from the checkpoint repository.
# (c) The Sorbian Institute releases -- one FLAC and one YAML per language --
# pinned by sha256 in training/speak-control/sources.json, cut into the 747
# utterances by data/scripts/prepare_sorbian_control.py.
import hashlib
run(sys.executable, "data/scripts/download_speech_data.py", "--split", "test")
TEST = CLONE / "data" / "speech" / "test.tsv"
n_test = sum(1 for _ in TEST.open(encoding="utf-8"))
print("test.tsv rows:", n_test)
if n_test != 925:
    raise SystemExit(f"FLEURS bs_ba test came back as {n_test} clips, not 925 -- not judging on a partial split")
from huggingface_hub import hf_hub_download
LIST = Path(hf_hub_download("rhasspy/piper-checkpoints", "sr/sr_RS/serbski_institut/medium/dataset.jsonl.gz",
                            repo_type="dataset", local_dir=str(SCRATCH / "sr-ckpt")))
sources = json.loads((CLONE / "training" / "speak-control" / "sources.json").read_text(encoding="utf-8"))
SORB = SCRATCH / "sorbian"
SORB.mkdir(parents=True, exist_ok=True)
got = {}
for name, src in sources["files"].items():
    target = SORB / name
    if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != src["sha256"]:
        run("curl", "-sSL", "-o", str(target), src["url"], quiet=True)
    digest = hashlib.sha256(target.read_bytes()).hexdigest()
    if digest != src["sha256"]:
        raise SystemExit(f"{name}: sha256 {digest}, the pre-registration pinned {src['sha256']} -- not the release the voice was cut from")
    got[name] = target
    print(f"  {name}: {target.stat().st_size / 1e6:.0f} MB, sha256 ok")
DATA = SCRATCH / "sorbian-data"
run(sys.executable, "data/scripts/prepare_sorbian_control.py", "--list", str(LIST),
    "--dsb-flac", str(got["dsb.flac"]), "--dsb-yaml", str(got["dsb.yaml"]),
    "--hsb-flac", str(got["hsb.flac"]), "--hsb-yaml", str(got["hsb.yaml"]), "--out", str(DATA))
CSV = DATA / "metadata.csv"
manifest = json.loads(Path(str(CSV) + ".manifest.json").read_text(encoding="utf-8"))
print("rows:", manifest["rows"], "| minutes:", manifest["minutes"], "| speakers:", manifest["speakers"],
      "| phoneme agreement:", manifest["phoneme_agreement"])
if manifest["rows"] != 747 or manifest["speakers"] != {"dsb": 408, "hsb": 339}:
    raise SystemExit(f"{manifest['rows']} rows / {manifest['speakers']} -- not the voice's own 747")
OFF.metric("train_rows", manifest["rows"], stage="data")
for voice, rec in manifest["phoneme_agreement"].items():
    OFF.metric(f"phoneme_agreement_{voice}", rec["share"], stage="data")


In [ ]:
# 6. The judge: the listener the app ships, from the lilly-listen-large-v3
# dataset, checked by fingerprint before a clip is heard. The voice is judged
# by the ear the product has, and by no other.
import hashlib

def fingerprint(build):
    # identical to scripts/fetch_models.listen_fingerprint and speech_bench.fingerprint
    h = hashlib.md5()
    for name in sorted(p.name for p in build.iterdir()
                       if p.is_file() and p.name != "dataset-metadata.json"):
        h.update(name.encode())
        h.update((build / name).read_bytes())
    return h.hexdigest()[:16]

INPUT = Path("/kaggle/input")
LISTEN = CLONE / "models" / "lilly" / "listen"
if LISTEN.exists():
    shutil.rmtree(LISTEN)
dirs = sorted(p.parent for p in INPUT.rglob("built.json")
              if (p.parent / "model.bin").is_file()) if INPUT.is_dir() else []
larges = [d for d in dirs
          if json.loads((d / "built.json").read_text()).get("base") == "openai/whisper-large-v3"]
if len(larges) != 1:
    raise SystemExit(f"need exactly one whisper-large-v3 listener attached (dataset "
                     f"lilly-listen-large-v3), found {larges}. Relaunch: python3 scripts/kaggle_train.py speak-bs")
shutil.copytree(larges[0], LISTEN, ignore=shutil.ignore_patterns("dataset-metadata.json"))
LISTEN_FP = fingerprint(LISTEN)
print("listener:", LISTEN, LISTEN_FP)
if LISTEN_FP != "e6bb58483586b06c":
    raise SystemExit(f"listener fingerprint {LISTEN_FP} is not the shipped e6bb58483586b06c -- "
                     "the voice would be judged by another ear")
# app.speech puts the listener on CPU int8 -- the product's path. The judgment is
# about which words are heard, not the chip; float16 on the T4 is what makes
# three voices x 167 sentences + 200 clips fit the session, as the instrument did.
GPU_ENV = {"LILLY_SPEECH_DEVICE": "cuda", "LILLY_SPEECH_COMPUTE": "float16",
           "LILLY_IGNORE_GUARD": "1", "PYTHONUNBUFFERED": "1"}


In [ ]:
# 7. The voice the app speaks with today -- the "before". Fetched the way every
# install fetches it, and checked to be the bytes the Mac measured 27% with.
run(sys.executable, "scripts/fetch_speak_bs.py")
BEFORE = CLONE / "models" / "lilly" / "speak-bs" / "voice.onnx"
before_md5 = hashlib.md5(BEFORE.read_bytes()).hexdigest()
print("before voice:", BEFORE, before_md5)
if before_md5 != "02c6e27ac7b4dfa84272df89edca9feb":
    raise SystemExit(f"the fetched sr_RS voice is {before_md5}, not the bytes the app ships "
                     "(02c6e27ac7b4dfa84272df89edca9feb)")


In [ ]:
# 8. The starting point: Piper's sr_RS checkpoint (924 MB), by md5, and a count
# of what the warm start will copy. 804 of 804 tensors match this trainer's
# model for a 2-speaker voice; with more speakers only the speaker table restarts.
from huggingface_hub import hf_hub_download
CKPT = Path(hf_hub_download("rhasspy/piper-checkpoints",
                            "sr/sr_RS/serbski_institut/medium/epoch=1899-step=178600.ckpt",
                            repo_type="dataset", local_dir=str(SCRATCH / "sr-ckpt")))
ckpt_md5 = hashlib.md5(CKPT.read_bytes()).hexdigest()
print("checkpoint:", CKPT, ckpt_md5)
if ckpt_md5 != "3dd3439e5c550d8201a9e7a2b1300a5b":
    raise SystemExit(f"sr_RS checkpoint is {ckpt_md5}, not the one the pre-registration names")
from piper.train.vits.lightning import VitsModel
old = torch.load(CKPT, map_location="cpu", weights_only=False)["state_dict"]
new = VitsModel(num_speakers=2, gin_channels=512, sample_rate=22050, num_symbols=256,
                batch_size=8, mos_metric=None).state_dict()
copied = sum(1 for k, v in old.items() if k in new and new[k].shape == v.shape)
print(f"warm start would copy {copied} of {len(old)} tensors into a 2-speaker model")
if copied != len(old) or copied < 800:
    raise SystemExit(f"only {copied} of {len(old)} tensors match -- this trainer does not read that checkpoint")
del old, new


In [ ]:
# 11. THREE ARMS, 3,000 steps each, from the same checkpoint on the same 747
# utterances. training/train_piper.py, fp32, 2 speakers so the warm start
# copies the speaker table too. Batch 2, not 8: the voice's own utterances run
# to 56 s and VITS pays for the whole padded batch (batch 16 filled the T4 on
# 20-second clips). Each arm has its own cache: Piper keys the cache on text,
# and sr and bs phonemize the same text differently.
import csv as _csv
PIPER = SCRATCH / "piper"
PIPER.mkdir(parents=True, exist_ok=True)
ARMS = {
    "A": ("sr", []),
    "B": ("bs", []),
    "C": ("bs", ["--model.learning_rate", "2e-5", "--model.learning_rate_d", "1e-5",
                 "--model.warmup_epochs", "2"]),
}
STEPS_EACH, MAX_TIME_EACH = "3000", "00:01:15:00"
trained = {}
for arm, (espeak_voice, extra) in ARMS.items():
    RUN = PIPER / f"run-{arm}"
    run(sys.executable, "training/train_piper.py", "fit",
        "--data.csv_path", str(CSV), "--data.cache_dir", str(PIPER / f"cache-{arm}"),
        "--data.config_path", str(PIPER / f"config-{arm}.json"), "--data.voice_name", f"control-{arm}",
        "--data.espeak_voice", espeak_voice, "--data.batch_size", "2", "--data.validation_split", "0.02",
        "--data.num_test_examples", "0", "--data.num_workers", "2",
        "--model.sample_rate", "22050", "--model.num_speakers", "2",
        "--model.gin_channels", "512", "--model.warmstart_ckpt", str(CKPT), "--model.mos_metric", "none",
        *extra,
        "--trainer.accelerator", "gpu", "--trainer.devices", "1", "--trainer.precision", "32",
        "--trainer.max_steps", STEPS_EACH, "--trainer.max_time", MAX_TIME_EACH,
        "--trainer.default_root_dir", str(RUN), "--trainer.check_val_every_n_epoch", "10",
        "--trainer.log_every_n_steps", "25", "--trainer.enable_progress_bar", "false",
        "--trainer.logger", "lightning.pytorch.loggers.CSVLogger",
        "--trainer.logger.save_dir", str(RUN), "--trainer.logger.name", "logs",
        env={"PYTHONUNBUFFERED": "1", "PYTORCH_ALLOC_CONF": "expandable_segments:True"})
    lasts = sorted(RUN.rglob("last.ckpt"))
    if len(lasts) != 1:
        raise SystemExit(f"arm {arm}: expected one last.ckpt under {RUN}, found {lasts}")
    state = torch.load(lasts[0], map_location="cpu", weights_only=False)
    steps = int(state["global_step"])
    bad = [k for k, v in state["state_dict"].items()
           if k.startswith("model_g.") and torch.is_tensor(v) and not torch.isfinite(v).all()]
    del state
    if bad:
        raise SystemExit(f"arm {arm}: non-finite weights: {bad[:5]}")
    if steps < 2500:
        raise SystemExit(f"arm {arm}: {steps} steps -- the wall cut it short of the pre-registered 3,000")
    metrics_files = sorted(RUN.rglob("metrics.csv"))
    if not metrics_files:
        raise SystemExit(f"arm {arm}: no metrics.csv")
    losses = {}
    for row in _csv.DictReader(metrics_files[0].open(encoding="utf-8")):
        for k, v in row.items():
            if v not in (None, "") and k.startswith(("loss", "train_", "val_")):
                f = float(v)
                if f != f or f in (float("inf"), float("-inf")):
                    raise SystemExit(f"arm {arm}: {k} is non-finite at step {row.get('step')}")
                losses[k] = f
    shutil.copy(metrics_files[0], f"/kaggle/working/metrics-{arm}.csv")
    CAND = SCRATCH / "cand" / arm
    CAND.mkdir(parents=True, exist_ok=True)
    run(sys.executable, "training/export_piper_onnx.py", "--checkpoint", str(lasts[0]),
        "--output-file", str(CAND / "voice.onnx"))
    shutil.copy(PIPER / f"config-{arm}.json", CAND / "voice.onnx.json")
    import wave
    from piper import PiperVoice, SynthesisConfig
    voice = PiperVoice.load(CAND / "voice.onnx", CAND / "voice.onnx.json")
    probe_wav = SCRATCH / f"probe-{arm}.wav"
    with wave.open(str(probe_wav), "wb") as w:
        voice.synthesize_wav("Dobar dan, kako ste? Sastanak je sutra u devet.", w,
                             syn_config=SynthesisConfig(speaker_id=0))
    with wave.open(str(probe_wav)) as w:
        secs = w.getnframes() / w.getframerate()
    del voice
    if secs < 1.0:
        raise SystemExit(f"arm {arm}: the probe rendered in under a second -- collapsed")
    trained[arm] = {"steps": steps, "losses": losses, "onnx": CAND / "voice.onnx", "probe_seconds": round(secs, 2)}
    print(f"arm {arm}: {steps} steps, probe {secs:.2f} s, last losses {losses}")
    OFF.metric(f"steps_{arm}", steps, stage="train")
    for k, v in losses.items():
        OFF.metric(f"{k}_{arm}", v, stage="train")


In [ ]:
# 12. THE READING -- the FLEURS test prefix, the before voice (speaker 0 of the
# checkpoint as the app fetches it), speaker 0 of each arm, the human recordings.
TEST_JSON = Path("/kaggle/working/speak-control-test.json")
arm_args = []
for arm, rec in trained.items():
    arm_args += ["--voice", f"arm{arm}={rec['onnx']}:0"]
run(sys.executable, "training/evaluate_speak.py", "--tsv", str(TEST), "--clips", "first200",
    "--voice", f"before={BEFORE}:0", *arm_args, "--human", "--listener", str(LISTEN),
    "--json", str(TEST_JSON), env=GPU_ENV)
test = json.loads(TEST_JSON.read_text(encoding="utf-8"))
before, human = test["voices"]["before"], test["human"]
if test["n_clips"] != 200 or human["n_clips"] != 200:
    raise SystemExit(f"judged {test['n_clips']} clips, not the 200-clip prefix")
OFF.check_trainproof(TEE)
readings = {}
for arm in trained:
    v = test["voices"][f"arm{arm}"]
    p = test["paired"][f"arm{arm} vs before"]
    delta, pval = p["delta_points"], p["p"]
    verdict = ("sound" if delta < 5 else "BROKEN" if (delta >= 10 and pval < 0.05) else "inconclusive")
    readings[arm] = {"wer": v["wer"], "delta": delta, "p": pval, "verdict": verdict}
    OFF.metric(f"test_wer_{arm}", v["wer"], stage="judge")
    OFF.metric(f"delta_{arm}", delta, stage="judge")
    OFF.metric(f"verdict_{arm}", verdict, stage="judge")
OFF.metric("test_wer_before", before["wer"], stage="judge")
OFF.metric("test_wer_human", human["wer"], stage="judge")
lines = ["# The voice pipeline's control -- the reading", "",
         f"Kaggle, {GPU}. Pre-registered: PREREGISTRATION.md, 'v7 -- speak -- the control'.",
         f"Listener {LISTEN_FP}; test prefix: {test['n_clips']} clips, {test['n_sentences']} sentences; "
         f"747 utterances of the voice's own data; phoneme agreement {manifest['phoneme_agreement']}.", "",
         "| voice | word error | wrong / words | vs before | p | reading |", "|---|---|---|---|---|---|",
         f"| before (the checkpoint, speaker 0, as fetched) | {before['wer']:.1f}% | {before['edits']} / {before['words']} | | | |"]
for arm, (espeak_voice, extra) in ARMS.items():
    v, r = test["voices"][f"arm{arm}"], readings[arm]
    lines.append(f"| arm {arm} ({espeak_voice}{', gentle' if extra else ''}), {trained[arm]['steps']} steps | "
                 f"{v['wer']:.1f}% | {v['edits']} / {v['words']} | {r['delta']:+.2f} | {r['p']:.4f} | **{r['verdict']}** |")
lines += [f"| human recordings | {human['wer']:.1f}% | {human['edits']} / {human['words']} | | | |", "",
          "Reading rule (pre-registered): sound if the arm is under +5 points from the before voice; "
          "BROKEN if +10 or more at p < 0.05; inconclusive between.", ""]
REPORT = "\n".join(lines) + "\n"
Path("/kaggle/working/speak-control.md").write_text(REPORT, encoding="utf-8")
print(REPORT)


In [ ]:
# 13. Package the reading. A control has no voice zip, by construction.
RESULTS = Path("/kaggle/working/lilly-speak-control-results.zip")
with zipfile.ZipFile(RESULTS, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(TEST_JSON, "speak-control-test.json")
    z.write(Path(str(CSV) + ".manifest.json"), "metadata.csv.manifest.json")
    for arm in trained:
        z.write(f"/kaggle/working/metrics-{arm}.csv", f"metrics-{arm}.csv")
        z.write(SCRATCH / "cand" / arm / "voice.onnx.json", f"voice-{arm}.onnx.json")
    z.write("/kaggle/working/speak-control.md", "speak-control.md")
    z.write(TEE, "stdout.txt")
if RESULTS.stat().st_size < 20_000:
    raise SystemExit(f"results zip is {RESULTS.stat().st_size} bytes -- that is not a result")
OFF.finish("complete", [RESULTS.name])
out = list(Path("/kaggle/working").rglob("*"))
print(f"Output holds {len(out)} entries")
assert len(out) < 50, [str(x) for x in out[:50]]


**Status ERROR or CANCEL → nothing here is a result.** Read `stdout.txt`, fix the
cause, relaunch. Recovery is not success.

**On COMPLETE:** `scripts/kaggle_train.py speak-control --fetch`;
`lilly-speak-control-results.zip` goes to `training/speak-control/`; write
`training/RESULTS-speak-control.md` and the outcome under the pre-registration
**whichever way it fell**. Nothing is installed from this run. What the reading
means for the next voice line is written in the pre-registration, in advance.
